# LifeMate: bước 3, sinh dữ liệu tổng hợp bằng model mở trên Colab

Model **Qwen3-30B-A3B-Instruct-2507-FP8** (khoảng 31GB, chạy bằng vLLM ngay trên GPU, không cần API key, dữ liệu không rời Colab). Với mỗi mẫu train, model đọc context, history và dữ liệu các app của mẫu đó rồi viết yêu cầu nói mới kèm chuỗi tool đúng: một yêu cầu làm được, một yêu cầu nhiều việc hoặc có điều kiện, một yêu cầu thiếu thông tin (phải hỏi lại) và bản đủ thông tin của chính yêu cầu đó, thỉnh thoảng thêm một yêu cầu ngoài phạm vi.

Mỗi mẫu phải qua hai lớp kiểm tra:
1. Chạy thật trên `ToolEnv` của mẫu gốc: bỏ nếu có call lỗi, dùng ID mà tool đọc chưa trả về, hoặc sai loại.
2. **Kiểm tra chéo**: chính model đó, không được xem đáp án, giải lại yêu cầu theo đúng quy tắc của agent trên một `ToolEnv` mới. Chỉ giữ nếu nó làm lại đủ mọi call của đáp án và ra cùng state cuối.

Chuẩn bị: **Runtime → GPU A100**, trên Drive đã có `MyDrive/zalo_lifemate/train+public_test.zip`, secret `GH_TOKEN` như trước.

Kết quả ghi thẳng lên Drive `MyDrive/zalo_lifemate/synth/synth.jsonl` sau mỗi lô 32 mẫu gốc; bị ngắt thì chạy lại cell, nó bỏ qua mẫu đã làm.

In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [4]:
import os, json
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/zalo_lifemate"
REPO = "/content/zalo_lifemate"
token = userdata.get("GH_TOKEN")
if not os.path.isdir(REPO):
    !git clone -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git {REPO}
    !git -C {REPO} remote set-url origin https://github.com/FirstKhoi/zalo_lifemate.git
%cd {REPO}
!git pull -q https://{token}@github.com/FirstKhoi/zalo_lifemate.git main && git log --oneline -1
os.environ["PYTHONPATH"] = f"{REPO}/src"
if not os.path.isdir("train+public_test"):
    !unzip -q "{DRIVE}/train+public_test.zip" -d .
SYNTH = f"{DRIVE}/synth/synth.jsonl"
!mkdir -p "{DRIVE}/synth" out
!pip install -q vllm
# kiểm tra bộ lọc trên chính đáp án train (không cần model): gần như mọi mẫu phải qua
!python scripts/synth_data.py --data ./train+public_test --out /dev/null --check-gold
# nếu cell sau báo lỗi lạ (SyntaxError ở dòng !, lỗi import): Runtime → Restart session rồi chạy lại từ đầu

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/zalo_lifemate
1912ad5 (HEAD -> main, origin/main, origin/feat/local-scorer, origin/HEAD) Reword the real train requests for training, checked by solving them again
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompa

## 1. Chạy thử 20 mẫu gốc
Lần đầu tải model khoảng 31GB từ Hugging Face (vài phút). Dòng `KV cache` / `Maximum concurrency` của vLLM cho biết GPU chạy song song được bao nhiêu yêu cầu; dòng `[synth] ... eta` cho biết thời gian.

- Nếu báo hết bộ nhớ, hoặc chạy chậm: thêm `--kv-fp8` vào `EXTRA` (cache dạng fp8, chứa được khoảng gấp đôi).
- Nếu model không nạp được trên A100: đổi `MODEL = "Qwen/Qwen3-14B-FP8"` (nhỏ hơn, chất lượng kém hơn một chút).

Xong cell này thì **tải `synth.jsonl` và `synth.jsonl.done` trong `MyDrive/zalo_lifemate/synth/` về `runs/` trên máy, kèm dòng `eta`, và báo mình xem chất lượng** trước khi chạy hết.

In [5]:
MODEL, EXTRA = "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8", ""
SYN = f'python scripts/synth_data.py --data ./train+public_test --out "{SYNTH}" --model {MODEL} {EXTRA}'
LOG = r'2>&1 | tee out/synth.log | grep --line-buffered -E "^\[synth\]|KV cache|Maximum concurrency|ERROR|Error|Traceback"'
!{SYN} --ids-file splits/train2_ids.txt --limit 20 --chunk 20 {LOG}
!grep -A 40 Traceback out/synth.log | tail -45

[synth] 0 base samples to do (1545 done before), backend=vllm model=Qwen/Qwen3-30B-A3B-Instruct-2507-FP8
(EngineCore pid=5796) INFO 10-30 08:48:20 [utils.py:347] Using LBNHC KV cache layout.
(EngineCore pid=5796) INFO 10-30 08:49:35 [gpu_worker.py:759] Available KV cache memory: 6.16 GiB
(EngineCore pid=5796) INFO 10-30 08:49:35 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9500 is equivalent to --gpu-memory-utilization=0.9337 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9663. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=5796) INFO 10-30 08:49:35 [kv_cache_utils.py:2327] GPU KV cache size: 67,312 tokens, Maximum concurrency for 20,480 tokens per request: 3.29x
[synth] done: 0 samples kept from 0 base samples; rejected: []


In [6]:
# xem nhanh: số mẫu theo loại, lý do bị bỏ, và vài mẫu
from collections import Counter
rows = [json.loads(l) for l in open(SYNTH)]
done = [json.loads(l) for l in open(SYNTH + ".done")]
print(len(rows), "mẫu giữ lại từ", len(done), "mẫu gốc:", Counter(r["synth_type"] for r in rows))
print("bị bỏ:", Counter(w for d in done for w in d["rejected"]).most_common(12))
for r in rows[:8]:
    print(f"\n[{r['synth_type']}] {r['audio_text']}")
    for c in r["label"]["tool_calls"]:
        print("   ", c["name"], json.dumps(c["arguments"], ensure_ascii=False))

1451 mẫu giữ lại từ 1545 mẫu gốc: Counter({'act': 494, 'oos': 375, 'pair': 357, 'ask': 147, 'multi': 78})
bị bỏ: [('ask: cross-check disagrees', 1087), ('multi: cross-check disagrees', 896), ('act: cross-check disagrees', 870), ('pair: cross-check disagrees', 783), ('ask: ask needs an installed tool_name and exactly one of its parameters', 121), ('ask: not installed: contacts.search', 120), ('multi: sys.ask_missing_info must be the last call, and only in ask', 104), ('pair: not installed: contacts.search', 100), ('pair: ungrounded id ewallet.transfer.contact_id', 50), ('multi: not installed: weather.get_forecast', 39), ('multi: ungrounded id smart_home.climate_set.device_id', 35), ('ask: sys.ask_missing_info must be the last call, and only in ask', 35)]

[act] Giảm cái đèn phòng khách xuống bốn mươi phần trăm thôi.
    smart_home.light_set {"device_id": "light_living", "brightness": 40}

[pair] Gửi tin nhắn cho Hương là anh đi làm trễ, tối nay về muộn.
    chat_ott.list_chats {}
    ch

## 2. Chạy hết các mẫu train2 (cho thí nghiệm trên dev2)
1245 mẫu gốc; thời gian xem dòng `eta` của lần chạy thử. Bị ngắt (Colab hết phiên) thì chạy lại cell cài đặt rồi cell này, nó làm tiếp.

In [7]:
!{SYN} --ids-file splits/train2_ids.txt {LOG}
!grep -A 40 Traceback out/synth.log | tail -45

[synth] 0 base samples to do (1545 done before), backend=vllm model=Qwen/Qwen3-30B-A3B-Instruct-2507-FP8
(EngineCore pid=7734) INFO 10-30 08:52:32 [utils.py:347] Using LBNHC KV cache layout.
(EngineCore pid=7734) INFO 10-30 08:52:47 [gpu_worker.py:759] Available KV cache memory: 6.16 GiB
(EngineCore pid=7734) INFO 10-30 08:52:47 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9500 is equivalent to --gpu-memory-utilization=0.9337 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9663. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=7734) INFO 10-30 08:52:47 [kv_cache_utils.py:2327] GPU KV cache size: 67,312 tokens, Maximum concurrency for 20,480 tokens per request: 3.29x
[synth] done: 0 samples kept from 0 base samples; rejected: []


## 3. Sau này: các mẫu dev2 (chỉ cho model cuối cùng)
300 mẫu nữa, chỉ dùng khi train bản cuối trên toàn bộ train. Khi chấm dev2, `build_sft.py --ids-file splits/train2_ids.txt` tự bỏ các mẫu sinh từ dev2 nên chạy cell này sớm cũng không làm hỏng phép đo.

In [8]:
!{SYN} --ids-file splits/dev2_ids.txt {LOG}
!grep -A 40 Traceback out/synth.log | tail -45

[synth] 0 base samples to do (1545 done before), backend=vllm model=Qwen/Qwen3-30B-A3B-Instruct-2507-FP8
(EngineCore pid=8353) INFO 10-30 08:53:57 [utils.py:347] Using LBNHC KV cache layout.
(EngineCore pid=8353) INFO 10-30 08:54:12 [gpu_worker.py:759] Available KV cache memory: 6.16 GiB
(EngineCore pid=8353) INFO 10-30 08:54:12 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9500 is equivalent to --gpu-memory-utilization=0.9337 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9663. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=8353) INFO 10-30 08:54:12 [kv_cache_utils.py:2327] GPU KV cache size: 67,312 tokens, Maximum concurrency for 20,480 tokens per request: 3.29x
[synth] done: 0 samples kept from 0 base samples; rejected: []


## 4. Viết lại câu yêu cầu thật (paraphrase)
Với mỗi mẫu train, model viết 3 cách nói khác của **chính câu yêu cầu đó** và giữ nguyên đáp án gốc, nên nhãn đúng sẵn và tỉ lệ nhiều bước, có lịch sử, hỏi lại giống hệt dữ liệu thật. Một câu viết lại chỉ được giữ nếu:
1. các cụm mà đáp án chép nguyên văn từ câu gốc (nội dung tin nhắn, tên, từ khoá tìm kiếm) vẫn còn nguyên văn;
2. model, không được xem đáp án, giải câu viết lại ra đúng đáp án, hoặc ra y hệt như khi nó giải câu gốc (nghĩa không đổi, kể cả khi model giải sai câu gốc).

Kết quả ghi thẳng lên Drive `MyDrive/zalo_lifemate/synth/para.jsonl`. Bị ngắt thì chạy lại cell cài đặt rồi cell này, nó làm tiếp. Cell đầu chạy thử 20 mẫu và in vài cặp câu gốc / câu viết lại để xem.

In [9]:
from collections import Counter
MODEL, EXTRA = "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8", ""
PARA = f"{DRIVE}/synth/para.jsonl"
PAR = f'python scripts/paraphrase_data.py --data ./train+public_test --out "{PARA}" --model {MODEL} {EXTRA}'
PLOG = r'2>&1 | tee out/para.log | grep --line-buffered -E "^\[para\]|KV cache|Maximum concurrency|ERROR|Error|Traceback"'
!{PAR} --ids-file splits/train2_ids.txt --limit 20 --chunk 20 {PLOG}
!grep -A 40 Traceback out/para.log | tail -45
orig = {s["id"]: s["audio_text"] for s in map(json.loads, open("train+public_test/train/train.jsonl"))}
rows = [json.loads(l) for l in open(PARA)]
print(len(rows), "câu giữ lại:", Counter(r["check"] for r in rows))
for r in rows[:12]:
    print(f"\n{orig[r['id']]}\n -> {r['text']}")

(EngineCore pid=8920) INFO 10-30 08:55:18 [utils.py:347] Using LBNHC KV cache layout.
(EngineCore pid=8920) INFO 10-30 08:55:33 [gpu_worker.py:759] Available KV cache memory: 6.16 GiB
(EngineCore pid=8920) INFO 10-30 08:55:33 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9500 is equivalent to --gpu-memory-utilization=0.9337 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9663. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=8920) INFO 10-30 08:55:33 [kv_cache_utils.py:2327] GPU KV cache size: 67,312 tokens, Maximum concurrency for 20,480 tokens per request: 3.29x
[para] 20 base samples to do (0 done before), model=Qwen/Qwen3-30B-A3B-Instruct-2507-FP8
[para] 20/20 kept=33 3 min, eta 0 min
[para] done: 33 rewordings kept from 20 base samples; rejected: [('solver differs', 27)]
33 câu giữ lại: Counter({

Ổn thì chạy hết 1245 mẫu train2 (khoảng 2 giờ, xem dòng `eta`), rồi 300 mẫu dev2 (chỉ dùng cho model cuối cùng; khi chấm dev2, `build_sft.py --ids-file splits/train2_ids.txt` tự bỏ chúng). Sau đó chạy `colab_llm.ipynb` (lần `para1`).

In [ ]:
!{PAR} --ids-file splits/train2_ids.txt {PLOG}
!grep -A 40 Traceback out/para.log | tail -45
!{PAR} --ids-file splits/dev2_ids.txt {PLOG}
!grep -A 40 Traceback out/para.log | tail -45

(EngineCore pid=10384) INFO 10-30 08:59:18 [utils.py:347] Using LBNHC KV cache layout.
(EngineCore pid=10384) INFO 10-30 08:59:33 [gpu_worker.py:759] Available KV cache memory: 6.16 GiB
(EngineCore pid=10384) INFO 10-30 08:59:33 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9500 is equivalent to --gpu-memory-utilization=0.9337 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9663. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=10384) INFO 10-30 08:59:33 [kv_cache_utils.py:2327] GPU KV cache size: 67,312 tokens, Maximum concurrency for 20,480 tokens per request: 3.29x
[para] 1225 base samples to do (20 done before), model=Qwen/Qwen3-30B-A3B-Instruct-2507-FP8
[para] 32/1225 kept=44 4 min, eta 149 min
[para] 64/1225 kept=72 10 min, eta 179 min
[para] 96/1225 kept=111 15 min, eta 179 min
[para] 128/1225 

Ghi chú: `scripts/synth_data.py --backend claude` dùng Claude qua API thay cho model mở (cần secret `ANTHROPIC_API_KEY`, tính tiền theo token, không có bước kiểm tra chéo).